In [1]:
from mmpose.apis import MMPoseInferencer
from datetime import datetime
import subprocess
import os
import json
import glob
import cv2

# Only include the following code if you are running the complete cloned project
while os.getcwd().split('\\')[-1] != 'mmpose-synthetic-tune':
    os.chdir('..')

In [2]:
coco_cow_category = {
    "id": 1,
    "name": "cow",
    "supercategory": "",
    "keypoints": [
        "Back1",
        "Back2",
        "Back3",
        "Back4",
        "Head",
        "Nose",
        "Neck",
        "L_Shoulder",
        "L_Elbow",
        "L_F_Paw",
        "R_Shoulder",
        "R_Elbow",
        "R_F_Paw",
        "Belly",
        "L_Hip",
        "L_Knee",
        "L_H_Paw",
        "R_Hip",
        "R_Knee",
        "R_H_Paw"
    ],
    "skeleton": [
        [
            3,
            4
        ],
        [
            16,
            17
        ],
        [
            12,
            13
        ],
        [
            8,
            9
        ],
        [
            11,
            14
        ],
        [
            18,
            1
        ],
        [
            6,
            5
        ],
        [
            18,
            19
        ],
        [
            14,
            18
        ],
        [
            14,
            15
        ],
        [
            9,
            10
        ],
        [
            8,
            14
        ],
        [
            1,
            2
        ],
        [
            19,
            20
        ],
        [
            15,
            1
        ],
        [
            15,
            16
        ],
        [
            7,
            6
        ],
        [
            4,
            7
        ],
        [
            11,
            12
        ],
        [
            2,
            3
        ],
        [
            7,
            11
        ],
        [
            7,
            8
        ]
    ]
    }

In [3]:
class MMPoseModelCoach:
    command = r'.\\venv\\Scripts\\python.exe'
    script = 'mmpose/tools/train.py'
    
    detector_model = {  #rtmdet
        "det_model": 'mmdetection/configs/rtmdet/rtmdet_l_swin_b_p6_4xb16-100e_coco.py',
        "det_weights": 'checkpoints/rtmdet_l_swin_b_p6_4xb16-100e_coco-a1486b6f.pth'
    }

    def __init__(self, config=None, resume=True, work_dir=None, notes=''):
        self.creation_time = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

        self.config = config
        self.resume = '--resume' if resume == True else ''
        if work_dir is not None:
            self.work_dir = work_dir
            self.config_path = glob.glob(f'{self.work_dir}/*.py')[0]
        else:
            self.config_path = f'dataset-coco/custom-configs/{self.config}'
            self.work_dir = f'models/_train-{self.creation_time}-{notes}'

        self.args = [
            self.command,
            self.script,
            self.config_path,
            '--work-dir',
            self.work_dir,
            self.resume,
        ]

    def train(self):
        # for linux
        # subprocess.run(self.args)
        print(f"Command to train: {' '.join(self.args)}") # For windows

    def visualize_results(self, vis_input, model_ckpt=None, radius=4, thickness=1):
        if model_ckpt is None:
            checkpoint_path = glob.glob(f'{self.work_dir}/best*.pth')[0]
        else:
            checkpoint_path = f'{self.work_dir}/{model_ckpt}'
        
        poser_model = {
            "pose2d": self.config_path,
            "pose2d_weights": checkpoint_path
        }

        inferencer = MMPoseInferencer(**poser_model, **self.detector_model, device='cuda:0')

        self.input_path = vis_input
        self.output_path = f'{self.work_dir}/vis_results'

        result_generator = inferencer(
            self.input_path,
            radius=radius,
            thickness=thickness,
            vis_out_dir=self.output_path,
            draw_heatmap=True,
            det_cat_ids=5
        )

        self.results = [res for res in result_generator]
        return self.results
        
    def results_to_coco(self):
        if not hasattr(self, 'results'):
            print('Run visualize_results first')
            return
        
        # Creating images
        all_images = []
        if os.path.isdir(self.input_path):
            file_names = sorted(glob.glob(f'{self.input_path}/*'))
        else:
            file_names = [self.input_path]
            
        id = 0
        for file_name in file_names:
            img = cv2.imread(file_name)
            h, w = img.shape[:2]
            img_coco = {
                "id": id,
                "width": w,
                "height": h,
                "file_name": file_name.split('/')[-1],
                "license": 0,
                "flickr_url": "",
                "coco_url": "",
                "date_captured": 0
            }
            all_images.append(img_coco)
            id += 1
            
        
        # Creating annotations
        all_anns = []
        kp_groups = list(map(lambda r: r['predictions'][0][0]['keypoints'], self.results))
        kp_score_groups = list(map(lambda r: r['predictions'][0][0]['keypoint_scores'], self.results))
        id = 0
        for kps, scores in zip(kp_groups, kp_score_groups):
            # base for coco keypoint annotations
            ann_coco = {
                "id": id,
                "image_id": id,
                "category_id": 1,
                "segmentation": [],
                "area": 0,
                "bbox": [],
                "iscrowd": 0,
                "attributes": {
                    "occluded": False,
                    "keyframe": False
                },
                "keypoints": [],
                "num_keypoints": 0
            }
            
            # adding keypoints in coco format
            ann_kps = []
            visible_kps = []
            for kp, score in zip(kps, scores):
                visibility = 2 if score >= .3 else 0
                ann_kps.extend([*kp, visibility])
                if visibility == 2:
                    visible_kps.extend(kp)

            ann_coco['keypoints'] = ann_kps
            
            # Calculating bounding box
            visible_kp_xs = [visible_kps[i] for i in range(0, len(visible_kps), 2)]
            visible_kp_ys = [visible_kps[i] for i in range(1, len(visible_kps), 2)]
            bbox_x = min(visible_kp_xs)
            bbox_y = min(visible_kp_ys)
            bbox_w = max(visible_kp_xs) - bbox_x
            bbox_h = max(visible_kp_ys) - bbox_y
            ann_coco['bbox'] = [ bbox_x, bbox_y, bbox_w, bbox_h ]
            
            # Calculating area
            ann_coco['area'] = bbox_w * bbox_h
            all_anns.append(ann_coco)
            id += 1
            
        coco_dataset = {
            "images": all_images,
            "annotations": all_anns,
            "categories": [coco_cow_category]
        }
        
        with open(f'{self.output_path}/results_coco_{self.creation_time}.json', 'w') as f:
            json.dump(coco_dataset, f)
        
        return coco_dataset
        
        

### Train and Visualize

In [21]:
HRnetBase_20kp = MMPoseModelCoach(
    config='td-hm_hrnet-w48_8xb64-256x256-train_on_base_hrnet-20kp-smaple98.py',
    notes='20kp-on-base-hrnet-w48-sample98'
)

# print(' '.join(HRnetBase_20kp.args))
HRnetBase_20kp.train()

Command to train: .\\venv\\Scripts\\python.exe mmpose/tools/train.py dataset-coco/custom-configs/td-hm_hrnet-w48_8xb64-256x256-train_on_base_hrnet-20kp-smaple98.py --work-dir models/_train-2024-07-31_03-52-35-20kp-on-base-hrnet-w48-sample98 --resume


In [17]:
import json

test_file = r'C:\Users\a.goldani\projects\mmpose-synthetic-tune-datasets\cow-coco\annotations\20kp-sample98-test.json'
with open(test_file, 'r') as f:
    test_set = json.load(f)
    
images = test_set['images']
file_names = [fr'C:\Users\a.goldani\projects\mmpose-synthetic-tune-datasets\cow-coco\20kp-sample98-data\{x["file_name"]}' for x in images ]

In [18]:
HRnetBase_20kp.visualize_results(
    vis_input=file_names,
)

Loads checkpoint by local backend from path: models/_train-2024-07-31_01-34-13-20kp-on-apk10k-hrnet-w48-sample98\best_coco_AP_epoch_160.pth
Loads checkpoint by local backend from path: checkpoints/rtmdet_l_swin_b_p6_4xb16-100e_coco-a1486b6f.pth
07/31 01:40:25 - mmengine - INFO - the output image has been saved at models/_train-2024-07-31_01-34-13-20kp-on-apk10k-hrnet-w48-sample98/vis_results\0_img020.png
07/31 01:40:25 - mmengine - INFO - the output image has been saved at models/_train-2024-07-31_01-34-13-20kp-on-apk10k-hrnet-w48-sample98/vis_results\1_img001.png
07/31 01:40:25 - mmengine - INFO - the output image has been saved at models/_train-2024-07-31_01-34-13-20kp-on-apk10k-hrnet-w48-sample98/vis_results\6_img038.png
07/31 01:40:26 - mmengine - INFO - the output image has been saved at models/_train-2024-07-31_01-34-13-20kp-on-apk10k-hrnet-w48-sample98/vis_results\1_img120.png
07/31 01:40:26 - mmengine - INFO - the output image has been saved at models/_train-2024-07-31_01-34-13

[defaultdict(list,
             {'visualization': [],
              'predictions': [[{'keypoints': [[1171.25, 128.75],
                  [996.25, 128.75],
                  [608.75, 316.25],
                  [633.75, 216.25],
                  [821.25, 316.25],
                  [383.75, 291.25],
                  [708.75, 316.25],
                  [646.25, 403.75],
                  [646.25, 478.75],
                  [558.75, 603.75],
                  [683.75, 416.25],
                  [696.25, 528.75],
                  [546.25, 616.25],
                  [858.75, 366.25],
                  [1108.75, 316.25],
                  [1208.75, 366.25],
                  [1121.25, 553.75],
                  [1158.75, 303.75],
                  [1221.25, 366.25],
                  [333.75, 666.25]],
                 'keypoint_scores': [0.6699261665344238,
                  0.2637161314487457,
                  0.19203241169452667,
                  0.29034730792045593,
                  

### Visualize Only

In [15]:
import json

test_file = r'C:\Users\a.goldani\projects\mmpose-synthetic-tune-datasets\cow-coco\annotations\20kp-sample98-test.json'
with open(test_file, 'r') as f:
    test_set = json.load(f)
    
images = test_set['images']
file_names = [fr'C:\Users\a.goldani\projects\mmpose-synthetic-tune-datasets\cow-coco\20kp-sample98-data\{x["file_name"]}' for x in images ]

['C:\\Users\\a.goldani\\projects\\mmpose-synthetic-tune-datasets\\cow-coco\\20kp-sample98-data\\0_img020.png',
 'C:\\Users\\a.goldani\\projects\\mmpose-synthetic-tune-datasets\\cow-coco\\20kp-sample98-data\\1_img001.png',
 'C:\\Users\\a.goldani\\projects\\mmpose-synthetic-tune-datasets\\cow-coco\\20kp-sample98-data\\6_img038.png',
 'C:\\Users\\a.goldani\\projects\\mmpose-synthetic-tune-datasets\\cow-coco\\20kp-sample98-data\\1_img120.png',
 'C:\\Users\\a.goldani\\projects\\mmpose-synthetic-tune-datasets\\cow-coco\\20kp-sample98-data\\7_img112.png',
 'C:\\Users\\a.goldani\\projects\\mmpose-synthetic-tune-datasets\\cow-coco\\20kp-sample98-data\\4_img135.png',
 'C:\\Users\\a.goldani\\projects\\mmpose-synthetic-tune-datasets\\cow-coco\\20kp-sample98-data\\0_img010.png',
 'C:\\Users\\a.goldani\\projects\\mmpose-synthetic-tune-datasets\\cow-coco\\20kp-sample98-data\\1_img041.png',
 'C:\\Users\\a.goldani\\projects\\mmpose-synthetic-tune-datasets\\cow-coco\\20kp-sample98-data\\2_img054.png',
 

In [20]:
_20kp_on_ap10k = MMPoseModelCoach(
    work_dir='models/trained-20kp-on-ap10k'
)
_20kp_on_ap10k.visualize_results(
    # model_ckpt='epoch_210.pth',
    vis_input=file_names
)
#dataset = _20kp_on_ap10k.results_to_coco()

Loads checkpoint by local backend from path: models/trained-20kp-on-ap10k\best_coco_AP_epoch_110.pth
Loads checkpoint by local backend from path: checkpoints/rtmdet_l_swin_b_p6_4xb16-100e_coco-a1486b6f.pth
07/31 01:48:37 - mmengine - INFO - the output image has been saved at models/trained-20kp-on-ap10k/vis_results\0_img020.png
07/31 01:48:37 - mmengine - INFO - the output image has been saved at models/trained-20kp-on-ap10k/vis_results\1_img001.png
07/31 01:48:38 - mmengine - INFO - the output image has been saved at models/trained-20kp-on-ap10k/vis_results\6_img038.png
07/31 01:48:38 - mmengine - INFO - the output image has been saved at models/trained-20kp-on-ap10k/vis_results\1_img120.png
07/31 01:48:39 - mmengine - INFO - the output image has been saved at models/trained-20kp-on-ap10k/vis_results\7_img112.png
07/31 01:48:39 - mmengine - INFO - the output image has been saved at models/trained-20kp-on-ap10k/vis_results\4_img135.png
07/31 01:48:39 - mmengine - INFO - the output imag

[defaultdict(list,
             {'visualization': [],
              'predictions': [[{'keypoints': [[383.75, 328.75],
                  [521.25, 316.25],
                  [621.25, 316.25],
                  [733.75, 316.25],
                  [871.25, 303.75],
                  [921.25, 403.75],
                  [758.75, 391.25],
                  [646.25, 491.25],
                  [608.75, 566.25],
                  [546.25, 628.75],
                  [696.25, 491.25],
                  [721.25, 591.25],
                  [746.25, 666.25],
                  [546.25, 541.25],
                  [433.75, 478.75],
                  [446.25, 541.25],
                  [508.75, 628.75],
                  [383.75, 478.75],
                  [321.25, 566.25],
                  [333.75, 666.25]],
                 'keypoint_scores': [0.9931684732437134,
                  0.48374950885772705,
                  0.7976033091545105,
                  0.5589048862457275,
                  0.81106

In [18]:
pretrained_ap10k = MMPoseModelCoach(
    work_dir='models/pretrained-hrnet_w48_ap10k_256x256-d95ab412_20211029'
)
pretrained_ap10k.visualize_results(
    model_ckpt='hrnet_w48_ap10k_256x256-d95ab412_20211029.pth',
    vis_input=vis_input
)

Loads checkpoint by local backend from path: models/pretrained-hrnet_w48_ap10k_256x256-d95ab412_20211029/hrnet_w48_ap10k_256x256-d95ab412_20211029.pth
07/16 19:45:41 - mmengine - WARNING - dataset_meta are not saved in the checkpoint's meta data, load via config.
Loads checkpoint by local backend from path: checkpoints/rtmdet_l_swin_b_p6_4xb16-100e_coco-a1486b6f.pth
07/16 19:50:04 - mmengine - INFO - the output video has been saved at models/pretrained-hrnet_w48_ap10k_256x256-d95ab412_20211029/vis_results\5-trim-resize.mp4


[defaultdict(list,
             {'visualization': [],
              'predictions': [[{'keypoints': [[271.25, 341.25],
                  [258.75, 353.75],
                  [271.25, 391.25],
                  [296.25, 316.25],
                  [533.75, 303.75],
                  [346.25, 391.25],
                  [346.25, 453.75],
                  [333.75, 503.75],
                  [308.75, 403.75],
                  [321.25, 453.75],
                  [321.25, 491.25],
                  [496.25, 416.25],
                  [496.25, 453.75],
                  [496.25, 516.25],
                  [458.75, 403.75],
                  [508.75, 453.75],
                  [508.75, 491.25]],
                 'keypoint_scores': [0.6791799664497375,
                  0.7600197792053223,
                  0.6316624879837036,
                  0.6487411260604858,
                  0.7952685952186584,
                  0.8526488542556763,
                  0.8665416836738586,
                  0.